# 205 — Vectorize S2 Predictions

Vectorizes postprocessed S2 tiles per tile, then merges all tile polygons.
No MMU filter — minimum patch = 1 pixel (100m² at 10m resolution).
Generates centroids from merged output.
Run after `204_postprocess_sat.ipynb`.

In [ ]:
from config_s2 import *
from helpers_s2 import *
import time, gc
import numpy as np
import pandas as pd

print(f"Config loaded ✓  |  Thresholds: {PIXEL_THRESHOLDS}")

In [ ]:
# ── Build tile list ───────────────────────────────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

ref_thresh = PIXEL_THRESHOLDS[0]
pp_dir     = s2_pp_tile_dir(ref_thresh)
pp_files   = sorted(f for f in os.listdir(pp_dir) if f.endswith(".tif"))

# Resume — check if vector tiles already exist
vec_dir      = s2_vector_tile_dir(ref_thresh)
os.makedirs(vec_dir, exist_ok=True)
vec_existing = set(os.listdir(vec_dir))
remaining    = [f for f in pp_files
                if f.replace(f"_pp_t{ref_thresh}.tif", f"_poly_t{ref_thresh}.gpkg")
                not in vec_existing]

print(f"  PP tiles:  {len(pp_files)}")
print(f"  Done:      {len(pp_files) - len(remaining)}")
print(f"  Remaining: {len(remaining)}")

# Per-threshold pp tile counts
for thresh in PIXEL_THRESHOLDS:
    n = len([f for f in os.listdir(s2_pp_tile_dir(thresh)) if f.endswith(".tif")])
    print(f"  PP tiles t={thresh}: {n}")

In [ ]:
# ── Vectorize per tile ────────────────────────────────────────────────────────
print("── Vectorizing tiles ───────────────────────────────────────")

MIN_M2 = 100  # 1 pixel at 10m resolution
t0     = time.time()

for i, pp_fname in enumerate(remaining):
    remaining_count = len(remaining) - i - 1
    source_fname    = pp_fname.replace(f"_pp_t{ref_thresh}.tif", ".tif")

    for thresh in PIXEL_THRESHOLDS:
        pp_path = os.path.join(s2_pp_tile_dir(thresh),
                               source_fname.replace(".tif", f"_pp_t{thresh}.tif"))
        if not os.path.exists(pp_path):
            continue

        with rasterio.open(pp_path) as src:
            data          = src.read(1)
            ref_transform = src.transform
            ref_crs       = src.crs
            pixel_area_m2 = abs(src.transform[0] * src.transform[4])

        if int((data == 1).sum()) == 0:
            del data; gc.collect()
            continue

        # Vectorize
        gdf = vectorize_tile_s2(data.copy(), ref_transform, ref_crs,
                                pixel_area_m2, min_m2=MIN_M2)
        del data; gc.collect()

        if gdf is None:
            continue

        # Save polygon tile
        vec_dir   = s2_vector_tile_dir(thresh)
        os.makedirs(vec_dir, exist_ok=True)
        poly_path = os.path.join(vec_dir,
            source_fname.replace(".tif", f"_poly_t{thresh}.gpkg"))
        gdf[["patch_id", "area_m2", "area_ha", "geometry"]].to_file(
            poly_path, driver="GPKG")
        del gdf; gc.collect()

    # Progress
    elapsed = time.time() - t0
    done    = i + 1
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h\n")
    else:
        print(f"  [{i+1}/{len(remaining)}] {source_fname} — {remaining_count} left — ETA {eta/60:.1f}min")

print(f"\nDone. Time: {(time.time()-t0)/3600:.1f}h")

In [ ]:
# ── Merge all tile polygons ───────────────────────────────────────────────────
print("── Merging polygon tiles ───────────────────────────────────")

MIN_M2 = 100
suffix = s2_output_suffix()
t0     = time.time()

for thresh in PIXEL_THRESHOLDS:
    print(f"\n  t={thresh} …")

    vec_dir    = s2_vector_tile_dir(thresh)
    poly_files = sorted(
        os.path.join(vec_dir, f)
        for f in os.listdir(vec_dir)
        if f.endswith(".gpkg") and "_poly_" in f
    )

    if not poly_files:
        print(f"    [!] No polygon tiles found — run vectorize step first")
        continue

    # Load and concatenate
    gdfs = []
    for p in poly_files:
        try:
            gdf = gpd.read_file(p)
            if len(gdf) > 0: gdfs.append(gdf)
        except Exception as e:
            print(f"    [!] {os.path.basename(p)}: {e}")

    if not gdfs:
        print(f"    [!] No valid polygon tiles")
        continue

    merged = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    del gdfs; gc.collect()
    print(f"    Loaded: {len(merged):,} patches from {len(poly_files)} tiles")

    # Merge touching polygons at tile boundaries
    from shapely.ops import unary_union
    print(f"    Merging touching polygons …")
    merged_geom = unary_union(merged.geometry)
    exploded    = gpd.GeoDataFrame(
        geometry=gpd.GeoSeries([merged_geom]).explode(index_parts=False),
        crs=merged.crs
    ).reset_index(drop=True)
    del merged; gc.collect()

    exploded["area_m2"]  = exploded.geometry.area.round(2)
    exploded["area_ha"]  = (exploded["area_m2"] / 10000).round(4)
    exploded = exploded[exploded["area_m2"] >= MIN_M2].reset_index(drop=True)
    exploded["patch_id"] = exploded.index + 1
    merged = exploded
    del exploded; gc.collect()

    print(f"    Patches: {len(merged):,}  Area: {merged['area_ha'].sum():.2f} ha")

    # Save polygons
    poly_out = os.path.join(S2_PP_OUT_DIR,
        f"lupine_s2_polygons_{suffix}_{MODEL_NAME_S2}_t{thresh}.gpkg")
    merged[["patch_id", "area_m2", "area_ha", "geometry"]].to_file(
        poly_out, driver="GPKG")
    print(f"    Polygons  → {os.path.basename(poly_out)}")

    # Save centroids
    cents = merged.copy()
    cents["geometry"] = cents.geometry.centroid
    cent_out = os.path.join(S2_PP_OUT_DIR,
        f"lupine_s2_centroids_{suffix}_{MODEL_NAME_S2}_t{thresh}.gpkg")
    cents[["patch_id", "area_m2", "area_ha", "geometry"]].to_file(
        cent_out, driver="GPKG")
    print(f"    Centroids → {os.path.basename(cent_out)}")

    del merged, cents; gc.collect()

print(f"\n  Done. Time: {(time.time()-t0)/60:.1f}min")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("── Summary ──────────────────────────────────────────────────")
suffix = s2_output_suffix()
rows   = []

for thresh in PIXEL_THRESHOLDS:
    poly_out = os.path.join(S2_PP_OUT_DIR,
        f"lupine_s2_polygons_{suffix}_{MODEL_NAME_S2}_t{thresh}.gpkg")
    if not os.path.exists(poly_out):
        continue
    gdf = gpd.read_file(poly_out)
    rows.append({
        "Threshold": thresh,
        "Patches":   len(gdf),
        "Area (ha)": round(gdf["area_ha"].sum(), 2),
    })
    del gdf

if rows:
    print(pd.DataFrame(rows).to_string(index=False))